# Big Data Foundation – Day 3
## Hands-on Lab: E-Commerce Big Data Analytics with PySpark

**Learning goals**
- Start Apache Spark in Google Colab
- Create and explore a transaction dataset
- Perform cleaning, transformation and aggregation
- Compare Pandas and PySpark
- Demonstrate MapReduce using RDDs
- Process 1 million records
- Generate business insights and a visualization


In [ ]:
# Cell 1 – Install PySpark
!pip install -q pyspark


In [ ]:
# Cell 2 – Create Spark Session
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = SparkSession.builder \
    .appName("BigDataFoundation_Day3") \
    .master("local[*]") \
    .getOrCreate()

print("Spark Version:", spark.version)
print("Spark Session created successfully!")


## 1. Create a sample E-commerce dataset

We will create **10,000 transactions** inside the notebook, so no external CSV download is required.


In [ ]:
# Cell 3 – Generate 10,000 transactions
import random
from datetime import datetime, timedelta

products = [
    ("Laptop", "Electronics", 55000),
    ("Mobile", "Electronics", 25000),
    ("Headphones", "Electronics", 3000),
    ("Chair", "Furniture", 4500),
    ("Table", "Furniture", 8000),
    ("Shoes", "Fashion", 2500),
    ("Shirt", "Fashion", 1200),
    ("Watch", "Fashion", 5000)
]

cities = ["Chennai", "Coimbatore", "Erode", "Salem", "Madurai", "Trichy"]
start_date = datetime(2026, 1, 1)

data = []

for i in range(1, 10001):
    product, category, price = random.choice(products)
    city = random.choice(cities)
    quantity = random.randint(1, 5)
    date = start_date + timedelta(days=random.randint(0, 180))

    data.append((
        i,
        date.strftime("%Y-%m-%d"),
        product,
        category,
        city,
        quantity,
        price
    ))

columns = [
    "Order_ID", "Order_Date", "Product",
    "Category", "City", "Quantity", "Price"
]

df = spark.createDataFrame(data, columns)

print("Total records:", df.count())


In [ ]:
# Cell 4 – Explore the data
df.show(10, truncate=False)


In [ ]:
# Cell 5 – Inspect the schema
df.printSchema()


## 2. Transformation – Calculate Total Sales

**Business rule:**

`Total Sales = Quantity × Price`


In [ ]:
# Cell 6 – Create Total_Sales
sales_df = df.withColumn(
    "Total_Sales",
    col("Quantity") * col("Price")
)

sales_df.show(10)


## 3. City-wise Revenue

This demonstrates a common Big Data operation: **grouping and aggregation**.


In [ ]:
# Cell 7 – City-wise revenue
city_sales = (
    sales_df
    .groupBy("City")
    .agg(
        sum("Total_Sales").alias("Total_Revenue")
    )
    .orderBy(desc("Total_Revenue"))
)

city_sales.show()


In [ ]:
# Cell 8 – Category-wise revenue and units sold
category_sales = (
    sales_df
    .groupBy("Category")
    .agg(
        sum("Total_Sales").alias("Total_Revenue"),
        sum("Quantity").alias("Units_Sold")
    )
    .orderBy(desc("Total_Revenue"))
)

category_sales.show()


In [ ]:
# Cell 9 – Top 5 products
top_products = (
    sales_df
    .groupBy("Product")
    .agg(
        sum("Total_Sales").alias("Revenue"),
        sum("Quantity").alias("Units_Sold")
    )
    .orderBy(desc("Revenue"))
    .limit(5)
)

top_products.show()


In [ ]:
# Cell 10 – Monthly revenue
monthly_sales = (
    sales_df
    .withColumn("Month", month(to_date("Order_Date")))
    .groupBy("Month")
    .agg(sum("Total_Sales").alias("Revenue"))
    .orderBy("Month")
)

monthly_sales.show()


## 4. Pandas vs PySpark

The business question is the same in both cases. The important difference is the processing model:

- **Pandas:** commonly processes data in the memory of one machine.
- **PySpark:** designed for distributed processing and can scale across a cluster.


In [ ]:
# Cell 11 – Pandas comparison
import pandas as pd

pandas_df = df.toPandas()

print("Pandas result:")
print(
    pandas_df.groupby("City")["Quantity"]
    .sum()
    .sort_values(ascending=False)
)


In [ ]:
# Cell 12 – PySpark equivalent
print("PySpark result:")

(
    df.groupBy("City")
      .sum("Quantity")
      .orderBy(desc("sum(Quantity)"))
      .show()
)


## 5. MapReduce Concept Demo

We now connect today's practical work with the Hadoop concept from the previous classes.

**Map → Shuffle/Group → Reduce**


In [ ]:
# Cell 13 – MapReduce using an RDD
transactions = [
    ("Chennai", 5000),
    ("Coimbatore", 3000),
    ("Chennai", 2000),
    ("Erode", 4000),
    ("Coimbatore", 1000),
    ("Erode", 1500)
]

rdd = spark.sparkContext.parallelize(transactions)

map_stage = rdd.map(lambda x: (x[0], x[1]))

print("Mapped key-value pairs:")
print(map_stage.collect())

reduce_stage = map_stage.reduceByKey(lambda a, b: a + b)

print("\nReduced result:")
print(reduce_stage.collect())


### What happened?

```text
Input transactions
       ↓
     MAP
       ↓
(city, amount)
       ↓
SHUFFLE / GROUP
       ↓
same cities together
       ↓
    REDUCE
       ↓
city-wise total
```


## 6. Scaling Demo – 1 Million Records

We will generate one million transactions and run a Spark aggregation.

> Note: Google Colab has limited RAM. This is a classroom scalability demonstration, not a benchmark of a production cluster.


In [ ]:
# Cell 14 – Generate 1 million records
large_data = []

for i in range(1, 1_000_001):
    product, category, price = random.choice(products)
    city = random.choice(cities)
    quantity = random.randint(1, 5)

    large_data.append((
        i, product, category, city, quantity, price
    ))

large_columns = [
    "Order_ID", "Product", "Category",
    "City", "Quantity", "Price"
]

large_df = spark.createDataFrame(large_data, large_columns)

print("Large dataset created.")
print("Records:", large_df.count())


In [ ]:
# Cell 15 – Process the 1 million records
import time

start = time.time()

large_result = (
    large_df
    .withColumn(
        "Total_Sales",
        col("Quantity") * col("Price")
    )
    .groupBy("City")
    .agg(sum("Total_Sales").alias("Revenue"))
    .orderBy(desc("Revenue"))
)

large_result.show()

elapsed = time.time() - start
print(f"Processing time: {elapsed:.2f} seconds")


## 7. Visualization – City-wise Revenue


In [ ]:
# Cell 16 – Visualize revenue
import matplotlib.pyplot as plt

city_pd = city_sales.toPandas()

plt.figure(figsize=(9, 5))
plt.bar(city_pd["City"], city_pd["Total_Revenue"])
plt.title("City-wise Revenue")
plt.xlabel("City")
plt.ylabel("Revenue")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 8. Automatically Generate Business Insights


In [ ]:
# Cell 17 – Business insights
top_city = city_sales.first()
top_category = category_sales.first()
top_product = top_products.first()

total_revenue = sales_df.agg(
    sum("Total_Sales").alias("Total_Revenue")
).first()["Total_Revenue"]

print("===== BUSINESS INSIGHTS =====")
print(f"Total Revenue       : ₹{total_revenue:,.0f}")
print(f"Top Revenue City    : {top_city['City']} (₹{top_city['Total_Revenue']:,.0f})")
print(f"Top Category        : {top_category['Category']} (₹{top_category['Total_Revenue']:,.0f})")
print(f"Top Product         : {top_product['Product']} (₹{top_product['Revenue']:,.0f})")


# 🎯 Mini Challenge

Try solving these without copying the previous cells:

1. Find the **average order value**.
2. Find the **top 3 cities by units sold**.
3. Find the **most expensive product sold**.
4. Find the **average quantity per order for each category**.
5. Create a chart showing **category-wise revenue**.

### Suggested starting point

```python
sales_df.groupBy("Category").agg(...)
```


In [ ]:
# Cell 18 – Challenge solution (run after students attempt it)

# 1. Average order value
print("1. Average Order Value")
sales_df.agg(avg("Total_Sales").alias("Average_Order_Value")).show()

# 2. Top 3 cities by units sold
print("2. Top 3 Cities by Units Sold")
(
    sales_df.groupBy("City")
    .agg(sum("Quantity").alias("Units_Sold"))
    .orderBy(desc("Units_Sold"))
    .limit(3)
    .show()
)

# 3. Most expensive product
print("3. Most Expensive Product")
df.orderBy(desc("Price")).select("Product", "Category", "Price").limit(1).show()

# 4. Average quantity by category
print("4. Average Quantity by Category")
(
    sales_df.groupBy("Category")
    .agg(avg("Quantity").alias("Average_Quantity"))
    .orderBy(desc("Average_Quantity"))
    .show()
)

# 5. Category-wise chart
category_pd = category_sales.toPandas()

plt.figure(figsize=(8, 5))
plt.bar(category_pd["Category"], category_pd["Total_Revenue"])
plt.title("Category-wise Revenue")
plt.xlabel("Category")
plt.ylabel("Revenue")
plt.tight_layout()
plt.show()


# Key Takeaways

### Big Data pipeline demonstrated today

```text
Raw Data
   ↓
Spark DataFrame
   ↓
Transformation
   ↓
Aggregation
   ↓
Distributed Processing Concept
   ↓
Business Insights
   ↓
Visualization
```

### Concepts covered
- Apache Spark
- PySpark DataFrame
- Schema
- Transformation
- Aggregation
- GroupBy
- RDD
- MapReduce
- Pandas vs PySpark
- Scalability
- Business analytics

**Next step:** In a production environment, the same pipeline can read data from sources such as HDFS, cloud storage, databases, APIs, streaming systems, or Kafka instead of generating the data inside Colab.


In [ ]:
# Cell 19 – Stop Spark when finished
spark.stop()
print("Spark session stopped.")
